In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("dataset/train.csv")
test  = pd.read_csv("dataset/test.csv")

print("Train rows:", len(train))
print("Test rows:", len(test))
print(train.columns.tolist())
train.head(5)


Train rows: 75000
Test rows: 75000
['sample_id', 'catalog_content', 'image_link', 'price']


,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


In [ ]:
import pandas as pd

train = pd.read_csv("dataset/train.csv")   # change path if needed
test  = pd.read_csv("dataset/test.csv")

print("✅ Files loaded successfully!")


✅ Files loaded successfully!


In [ ]:
train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sample_id        75000 non-null  int64  
 1   catalog_content  75000 non-null  object 
 2   image_link       75000 non-null  object 
 3   price            75000 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 2.3+ MB


In [ ]:
train.describe()

,sample_id,price
count,75000.000000,75000.000000
mean,149841.917707,23.647654
std,86585.346513,33.376932
min,0.000000,0.130000
25%,73845.750000,6.795000
50%,150129.000000,14.000000
75%,225040.250000,28.625000
max,299438.000000,2796.000000


In [ ]:
train['catalog_content'].isna().mean()

np.float64(0.0)

In [ ]:
train['image_link'].isna().mean()

np.float64(0.0)

In [ ]:
train['price'].min(), train['price'].max(), train['price'].skew()

(np.float64(0.13), np.float64(2796.0), np.float64(13.601388975432753))

In [ ]:
import re
from html import unescape

def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s)
    s = unescape(s)                 # convert HTML entities
    s = s.replace('\n', ' ').replace('\r', ' ')
    s = re.sub(r'\s+', ' ', s)      # collapse whitespace
    s = s.strip()
    return s

train['raw_text'] = train['catalog_content'].apply(normalize_text)
test['raw_text']  = test['catalog_content'].apply(normalize_text)

# quick preview
train[['sample_id','raw_text']].head(5)


,sample_id,raw_text
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ..."
1,198967,"Item Name: Salerno Cookies, The Original Butte..."
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy..."
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun..."


In [ ]:
def simple_tokens(s):
    # split on spaces and punctuation, but keep units like "500ml" as one token
    # we'll split on whitespace and on slashes/commas/semicolons
    s2 = re.sub(r'[\/,;()\[\]\{\}]', ' ', s)
    toks = s2.split()
    return toks

train['tokens'] = train['raw_text'].apply(simple_tokens)


In [ ]:
import math

IPQ_PATTERNS = [
    r'(\d+)\s*(?:-?\s?pack|pack(?:s)?|pcs|pieces)\b',        # "6 pack", "2 packs", "3 pcs"
    r'\bpack\s*of\s*(\d+)\b',                               # "pack of 6"
    r'(\d+)\s*[x×\*]\s*\d+\s*(?:ml|l|g|kg|oz)?\b',          # "2 x 500ml", "2×500 ml"
    r'(\d+)\s*[x×\*]\s*[\d\.,]+\s*(?:ml|l|g|kg|oz)\b',      # variant
    r'(\d+)\s*[x×\*]\b',                                    # "2 x"
    r'(\d+)\s*(?:count)\b'                                  # "6 count"
]

def extract_ipq(text):
    t = text.lower()
    for pat in IPQ_PATTERNS:
        m = re.search(pat, t)
        if m:
            try:
                val = int(m.group(1))
                if val > 0 and val < 10000:
                    return val
            except:
                pass
    # fallback: check for tokens like "pack of 3 (3x200ml)" where the inner '3x' exists
    m2 = re.search(r'(\d+)\s*[x×]\s*\d+', t)
    if m2:
        try: return int(m2.group(1))
        except: pass
    return np.nan

train['ipq'] = train['raw_text'].apply(extract_ipq)
test['ipq']  = test['raw_text'].apply(extract_ipq)

train['ipq'].fillna(1, inplace=True)   # if not found, assume 1 (single item) — **document this assumption**
test['ipq'].fillna(1, inplace=True)


C:\Users\HP\AppData\Local\Temp\ipykernel_35652\389980550.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['ipq'].fillna(1, inplace=True)   # if not found, assume 1 (single item) — **document this assumption**
C:\Users\HP\AppData\Local\Temp\ipykernel_35652\389980550.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we

In [ ]:
train['ipq'].value_counts().head(20)
train.loc[train['raw_text'].str.contains('pack of', case=False), ['raw_text','ipq']].head(10)


,raw_text,ipq
0,"Item Name: La Victoria Green Taco Sauce Mild, ...",6.0
1,"Item Name: Salerno Cookies, The Original Butte...",4.0
2,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",6.0
6,Item Name: Goya Foods Sazonador Total Seasonin...,6.0
9,Item Name: Mrs. Miller's Seedless Black Raspbe...,4.0
11,"Item Name: Albanese Assorted Gummi Bears, Suga...",2.0
13,Item Name: Smuckers Natural Peanut Butter Chun...,12.0
14,Item Name: BODYARMOR LYTE Sports Drink Low-Cal...,12.0
16,Item Name: Himalania Pink Salt Fine Jar 10.0 O...,6.0
17,Item Name: BUSH'S BEST 16 oz Canned Barbecue B...,12.0


In [ ]:
UNIT_PAT = re.compile(r'(\d+(?:[.,]\d+)?)\s*(ml|l|g|kg|gram|grams|ounce|oz|pcs)\b', flags=re.I)

def extract_size(text):
    t = text.lower()
    m = UNIT_PAT.search(t)
    if m:
        num = m.group(1).replace(',', '.')
        try:
            val = float(num)
            unit = m.group(2).lower()
            # normalize to base units (ml for volume, g for weight)
            if unit in ['l']:
                val_ml = val * 1000.0
                return ('vol_ml', val_ml)
            if unit in ['ml']:
                return ('vol_ml', val)
            if unit in ['kg']:
                return ('wt_g', val * 1000.0)
            if unit in ['g','gram','grams']:
                return ('wt_g', val)
            if unit in ['oz','ounce']:
                return ('wt_g', val * 28.3495)
            if unit in ['pcs']:
                return ('pieces', val)
        except:
            pass
    return (None, np.nan)

train[['size_unit','size_val']] = train['raw_text'].apply(lambda s: pd.Series(extract_size(s)))
test[['size_unit','size_val']]  = test['raw_text'].apply(lambda s: pd.Series(extract_size(s)))

# fillna with sensible defaults
train['size_val'].fillna(0, inplace=True)
test['size_val'].fillna(0, inplace=True)


C:\Users\HP\AppData\Local\Temp\ipykernel_35652\210276645.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['size_val'].fillna(0, inplace=True)
C:\Users\HP\AppData\Local\Temp\ipykernel_35652\210276645.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when do

In [ ]:
def extract_brand_candidates(text):
    if not text: return (None, None)
    toks = text.split()
    # candidate 1: first token if it's alphabetical and not generic words
    first = toks[0].strip(' -,:;.')
    # candidate 2: first two tokens if both capitalized
    second = toks[1].strip(' -,:;.') if len(toks) > 1 else ""
    return first, second

train[['brand1','brand2']] = train['raw_text'].apply(lambda s: pd.Series(extract_brand_candidates(s)))
test[['brand1','brand2']] = test['raw_text'].apply(lambda s: pd.Series(extract_brand_candidates(s)))


In [ ]:
GENERIC = set(["pack", "new", "combo", "set", "original", "imported", "100%"])
def sanitize_brand_cand(x):
    if pd.isna(x) or x=="":
        return None
    low = x.lower()
    if low in GENERIC or re.search(r'\d', x):
        return None
    return x

train['brand1'] = train['brand1'].apply(sanitize_brand_cand)
train['brand2'] = train['brand2'].apply(sanitize_brand_cand)


In [ ]:
cand_counts = train['brand1'].value_counts().head(500)
cand_counts.head(50)


brand1
Item    75000
Name: count, dtype: int64

In [ ]:
import re

def simple_tokens(s):
    s2 = re.sub(r'[\/,;()\[\]\{\}]', ' ', s)
    toks = s2.split()
    return toks


In [ ]:
train['tokens'] = train['raw_text'].apply(simple_tokens)
test['tokens']  = test['raw_text'].apply(simple_tokens)


In [ ]:
train['text_len'] = train['raw_text'].str.len()
train['num_tokens'] = train['tokens'].apply(len)
train['num_digits'] = train['raw_text'].str.count(r'\d')


In [ ]:
print(train.columns)
print(train[['raw_text','tokens']].head(3))


Index(['sample_id', 'catalog_content', 'image_link', 'price', 'raw_text',
       'tokens', 'ipq', 'size_unit', 'size_val', 'brand1', 'brand2',
       'text_len', 'num_tokens', 'num_digits', 'upper_ratio', 'has_image',
       'brand_present'],
      dtype='object')
                                            raw_text  \
0  Item Name: La Victoria Green Taco Sauce Mild, ...   
1  Item Name: Salerno Cookies, The Original Butte...   
2  Item Name: Bear Creek Hearty Soup Bowl, Creamy...   

                                              tokens  
0  [Item, Name:, La, Victoria, Green, Taco, Sauce...  
1  [Item, Name:, Salerno, Cookies, The, Original,...  
2  [Item, Name:, Bear, Creek, Hearty, Soup, Bowl,...  


In [ ]:
# text metrics
train['text_len'] = train['raw_text'].str.len()
train['num_tokens'] = train['tokens'].apply(len)
train['num_digits'] = train['raw_text'].str.count(r'\d')
train['upper_ratio'] = train['catalog_content'].fillna('').str.count(r'[A-Z]') / (train['text_len'] + 1)

# flags
train['has_image'] = train['image_link'].notna().astype(int)
train['brand_present'] = train['brand1'].notna().astype(int)

# same for test
test['text_len'] = test['raw_text'].str.len()
test['num_tokens'] = test['tokens'].apply(len)
test['num_digits'] = test['raw_text'].str.count(r'\d')
test['upper_ratio'] = test['catalog_content'].fillna('').str.count(r'[A-Z]') / (test['text_len'] + 1)
test['has_image'] = test['image_link'].notna().astype(int)
test['brand_present'] = test['brand1'].notna().astype(int)


In [ ]:
train['per_item_size_ml'] = np.where(train['size_unit']=='vol_ml', train['size_val'] / train['ipq'], 0)
train['per_item_wt_g']   = np.where(train['size_unit']=='wt_g', train['size_val'] / train['ipq'], 0)
test['per_item_size_ml'] = np.where(test['size_unit']=='vol_ml', test['size_val'] / test['ipq'], 0)
test['per_item_wt_g']    = np.where(test['size_unit']=='wt_g', test['size_val'] / test['ipq'], 0)


In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm', disable=['parser','ner'])

def lemmatize_text(s, nlp=nlp):
    if not s:
        return ""
    doc = nlp(s)
    toks = [tok.lemma_.lower() for tok in doc if not tok.is_stop and not tok.is_punct and not tok.is_space]
    return " ".join(toks)


In [ ]:
import sys
print(sys.executable)


c:\Users\HP\smart-pricing\venv\Scripts\python.exe


In [ ]:
train.to_csv("processed/train_step1.csv", index=False)
test.to_csv("processed/test_step1.csv", index=False)


In [ ]:
train['ipq'].value_counts().head(20)


ipq
1.0      44006
6.0       5654
12.0      5334
2.0       4655
3.0       4023
4.0       2419
24.0      1421
8.0       1235
10.0       781
5.0        627
20.0       435
18.0       342
16.0       268
40.0       251
36.0       223
48.0       216
100.0      214
15.0       202
30.0       193
50.0       167
Name: count, dtype: int64

In [ ]:
train.isna().sum().sort_values(ascending=False).head(20)


size_unit           27268
catalog_content         0
sample_id               0
image_link              0
price                   0
tokens                  0
raw_text                0
ipq                     0
size_val                0
brand1                  0
brand2                  0
text_len                0
num_tokens              0
num_digits              0
upper_ratio             0
has_image               0
brand_present           0
per_item_size_ml        0
per_item_wt_g           0
dtype: int64

In [ ]:
bad_ipq = train[train['ipq'] == 1].sample(20)  # contains many single-pack items; inspect some
bad_ipq[['raw_text','ipq']].head(20)


,raw_text,ipq
72212,"Item Name: Davidson's Organics, Decaffeinated ...",1.0
7542,Item Name: Blue Diamond Lightly Salted Almonds...,1.0
52235,Item Name: Land O Lakes Cocoa Classics Arctic ...,1.0
28106,Item Name: Asturi Bruschettini Classico Virgin...,1.0
51059,Item Name: Ferrero Rocher Premium Chocolate Ba...,1.0
45469,Item Name: Moringa Seeds Kernel Shelled 2 lbs....,1.0
16092,"Item Name: Gilan Dried Lime Slices, All Natura...",1.0
21970,"Item Name: Wolfgang Puck Coffee, Chef's Reserv...",1.0
33065,Item Name: Dependable Food Onion Soup Base - O...,1.0
45727,Item Name: Wallacea Coffee Certified Wild Kopi...,1.0


In [ ]:
train.groupby('ipq')['price'].median().sort_values(ascending=False).head(20)


ipq
1800.0    173.240
213.0     152.250
168.0     105.990
755.0     104.990
78.0      102.485
384.0      99.165
175.0      95.675
9000.0     95.290
6000.0     93.510
420.0      91.550
720.0      84.730
255.0      79.990
1900.0     65.290
1212.0     64.450
528.0      61.240
164.0      59.990
900.0      56.280
780.0      55.400
297.0      54.990
416.0      54.990
Name: price, dtype: float64

In [ ]:
train['ipq_extracted'] = (train['ipq'] > 1).astype(int)
train['size_extracted'] = (~train['size_val'].isna() & (train['size_val']>0)).astype(int)


In [ ]:
import os

# ensure processed folder exists
os.makedirs("processed", exist_ok=True)

train_proc.to_csv("processed/train_step1.csv", index=False)
test_proc.to_csv("processed/test_step1.csv", index=False)


In [ ]:
for col in train_proc.columns:
    if str(train_proc[col].dtype) == 'period[M]':
        train_proc[col] = train_proc[col].astype(str)


In [ ]:
print(train_proc.dtypes)


sample_id             int64
catalog_content      object
image_link           object
price               float64
raw_text             object
tokens               object
ipq                   int64
size_unit            object
size_val            float64
brand1               object
brand2               object
text_len              int64
num_tokens            int64
num_digits            int64
upper_ratio         float64
has_image             int64
brand_present         int64
per_item_size_ml    float64
per_item_wt_g       float64
ipq_extracted         int64
size_extracted        int64
dtype: object


In [ ]:
for col in train_proc.columns:
    if pd.api.types.is_list_like(train_proc[col].iloc[0]):
        train_proc[col] = train_proc[col].astype(str)  # convert lists to strings
    elif pd.api.types.is_integer_dtype(train_proc[col]):
        train_proc[col] = train_proc[col].fillna(0).astype(int)
    elif pd.api.types.is_float_dtype(train_proc[col]):
        train_proc[col] = train_proc[col].fillna(0.0).astype(float)
    else:
        train_proc[col] = train_proc[col].astype(str)


In [ ]:
import os
os.makedirs("processed", exist_ok=True)

train_proc.to_csv("processed/train_step1.csv", index=False)
test_proc.to_csv("processed/test_step1.csv", index=False)


In [ ]:
train_proc_simple = train_proc.select_dtypes(include=['int', 'float', 'object'])
train_proc_simple.to_csv("processed/train_step1.csv", index=False)


In [ ]:
import os
import pandas as pd

# ensure folder exists
os.makedirs("processed", exist_ok=True)

# convert all columns to CSV-safe types
for col in train_proc.columns:
    if pd.api.types.is_integer_dtype(train_proc[col]):
        train_proc[col] = train_proc[col].fillna(0).astype(int)
    elif pd.api.types.is_float_dtype(train_proc[col]):
        train_proc[col] = train_proc[col].fillna(0.0).astype(float)
    else:
        # everything else to string
        train_proc[col] = train_proc[col].astype(str)

for col in test_proc.columns:
    if pd.api.types.is_integer_dtype(test_proc[col]):
        test_proc[col] = test_proc[col].fillna(0).astype(int)
    elif pd.api.types.is_float_dtype(test_proc[col]):
        test_proc[col] = test_proc[col].fillna(0.0).astype(float)
    else:
        test_proc[col] = test_proc[col].astype(str)


In [ ]:
train_proc.to_csv("processed/train_step1.csv", index=False)
test_proc.to_csv("processed/test_step1.csv", index=False)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
# Example: using the cleaned text from Step 1
train_text = train['clean_text_simple']  # cleaned text column
test_text  = test['clean_text_simple']


In [ ]:
print(train.columns)
print(test.columns)


Index(['sample_id', 'catalog_content', 'image_link', 'price', 'raw_text',
       'tokens', 'ipq', 'size_unit', 'size_val', 'brand1', 'brand2',
       'text_len', 'num_tokens', 'num_digits', 'upper_ratio', 'has_image',
       'brand_present', 'per_item_size_ml', 'per_item_wt_g', 'ipq_extracted',
       'size_extracted'],
      dtype='object')
Index(['sample_id', 'catalog_content', 'image_link', 'raw_text', 'ipq',
       'size_unit', 'size_val', 'brand1', 'brand2', 'text_len', 'tokens',
       'num_tokens', 'num_digits', 'upper_ratio', 'has_image', 'brand_present',
       'per_item_size_ml', 'per_item_wt_g'],
      dtype='object')


In [ ]:
# Create a cleaned text column from the original catalog_content
train['clean_text_simple'] = train['catalog_content'].astype(str).str.lower()
test['clean_text_simple']  = test['catalog_content'].astype(str).str.lower()


In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,     # limit vocabulary to top 10k words
    ngram_range=(1,2),      # include unigrams and bigrams
    min_df=5,               # ignore words that appear in fewer than 5 documents
    max_df=0.8               # ignore words appearing in more than 80% of documents
)


In [ ]:
# Fit on training text
tfidf.fit(train_text)

# Transform training and test data
X_train_tfidf = tfidf.transform(train_text)
X_test_tfidf  = tfidf.transform(test_text)

print("Shape of training TF-IDF:", X_train_tfidf.shape)
print("Shape of test TF-IDF:", X_test_tfidf.shape)


Shape of training TF-IDF: (75000, 10000)
Shape of test TF-IDF: (75000, 10000)


In [ ]:
import pickle

# Save vectorizer for later use
with open("processed/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)


In [ ]:
import os

os.makedirs("images/train", exist_ok=True)
os.makedirs("images/test", exist_ok=True)


In [ ]:
import requests
from tqdm import tqdm
from PIL import Image
from io import BytesIO

def download_image(url, save_path):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # check for HTTP errors
        img = Image.open(BytesIO(response.content)).convert('RGB')  # force 3 channels
        img.save(save_path)
        return True
    except Exception as e:
        print(f"Failed to download {url}: {e}")
        return False


In [ ]:
import os
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# === Configurations ===
SAVE_DIR = "images/train"      # folder to save images
FAILED_LOG = "failed_urls.txt" # log failed downloads
RETRIES = 3                     # retry attempts
MAX_THREADS = 10                # number of parallel downloads

os.makedirs(SAVE_DIR, exist_ok=True)

# === Helper function to download one image ===
def download_image(row):
    url = row['image_link']
    sample_id = row['sample_id']
    save_path = os.path.join(SAVE_DIR, f"{sample_id}.jpg")

    # Skip if already downloaded
    if os.path.exists(save_path):
        return True

    # Try downloading with retries
    for attempt in range(RETRIES):
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            img = Image.open(BytesIO(response.content)).convert('RGB')
            img.save(save_path)
            return True
        except Exception as e:
            if attempt == RETRIES - 1:
                with open(FAILED_LOG, "a", encoding="utf-8") as f:
                    f.write(url + "\n")
                return False

# === Run parallel downloads ===
with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    futures = {executor.submit(download_image, row): row for idx, row in train.iterrows()}
    for _ in tqdm(as_completed(futures), total=len(futures)):
        pass  # progress bar only


100%|██████████| 75000/75000 [12:47<00:00, 97.73it/s]   


In [ ]:
import pandas as pd

# Load failed URLs
with open("failed_urls.txt", "r", encoding="utf-8") as f:
    failed_urls = [line.strip() for line in f.readlines()]

print("Total failed URLs to retry: {}".format(len(failed_urls)))
 

Total failed URLs to retry: 1


In [ ]:
# Filter the train DataFrame to only include failed URLs
failed_df = train[train['image_link'].isin(failed_urls)].copy()
print(f"Images to retry: {len(failed_df)}")


Images to retry: 1


In [ ]:
import os
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

SAVE_DIR = "images/train"      # Folder where images are saved
RETRIES = 3                     # Retry count per image
MAX_THREADS = 10                # Number of parallel downloads
FAILED_LOG = "failed_urls_retry.txt"  # Log failed URLs after retry

os.makedirs(SAVE_DIR, exist_ok=True)

def download_image(row):
    url = row['image_link']
    sample_id = row['sample_id']
    save_path = os.path.join(SAVE_DIR, f"{sample_id}.jpg")

    # Skip if already exists
    if os.path.exists(save_path):
        return True

    for attempt in range(RETRIES):
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            img = Image.open(BytesIO(response.content)).convert("RGB")
            img.save(save_path)
            return True
        except Exception as e:
            if attempt == RETRIES - 1:
                with open(FAILED_LOG, "a", encoding="utf-8") as f:
                    f.write(url + "\n")
                return False

# Run parallel download
with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    futures = {executor.submit(download_image, row): row for idx, row in failed_df.iterrows()}
    for _ in tqdm(as_completed(futures), total=len(futures)):
        pass


100%|██████████| 1/1 [00:00<00:00,  2.64it/s]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
print(train.columns)


Index(['sample_id', 'catalog_content', 'image_link', 'price', 'raw_text',
       'tokens', 'ipq', 'size_unit', 'size_val', 'brand1', 'brand2',
       'text_len', 'num_tokens', 'num_digits', 'upper_ratio', 'has_image',
       'brand_present'],
      dtype='object')


In [ ]:
import re

def clean_text_simple(text):
    # Lowercase
    text = text.lower()
    # Remove special characters and numbers (keep words only)
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply to train and test
train['clean_text_simple'] = train['raw_text'].apply(clean_text_simple)
test['clean_text_simple']  = test['raw_text'].apply(clean_text_simple)


In [ ]:
train_text = train['clean_text_simple']
test_text  = test['clean_text_simple']


In [ ]:
import pandas as pd
from scipy import sparse
import joblib

# Load preprocessed data
train = pd.read_csv("processed/train_step1.csv")
test  = pd.read_csv("processed/test_step1.csv")

# Load the trained TF-IDF vectorizer
vectorizer = joblib.load("processed/tfidf_vectorizer.pkl")

# The column that contains your cleaned text
text_col = "raw_text"   # change to 'clean_text_simple' if you have that column

# Transform text into TF-IDF features
tfidf_train = vectorizer.transform(train[text_col].fillna(""))
tfidf_test  = vectorizer.transform(test[text_col].fillna(""))

# Save them as sparse matrices
sparse.save_npz("processed/tfidf_train.npz", tfidf_train)
sparse.save_npz("processed/tfidf_test.npz", tfidf_test)

print("✅ TF-IDF matrices saved successfully!")


✅ TF-IDF matrices saved successfully!


In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, hstack


In [ ]:
# Load metadata/text data
train_df = pd.read_csv("processed/train_step1.csv")
test_df = pd.read_csv("processed/test_step1.csv")

# Load TF-IDF text features
X_train_tfidf = load_npz("processed/tfidf_train.npz")
X_test_tfidf = load_npz("processed/tfidf_test.npz")


In [ ]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())


2.5.1+cu121
CUDA available: True


In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision import models


In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import requests
from io import BytesIO
import torch
import torchvision.transforms as transforms
from torchvision import models


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet = models.resnet50(weights="IMAGENET1K_V1")
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])  # remove last FC layer
resnet.eval()
resnet.to(device)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:04<00:00, 22.3MB/s]


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def get_embedding(url):
    try:
        response = requests.get(url, timeout=5)
        image = Image.open(BytesIO(response.content)).convert('RGB')
        image = transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            features = resnet(image).squeeze().cpu().numpy()
        return features
    except Exception as e:
        return np.zeros(2048)  # fallback vector for failed downloads


In [ ]:
train_df = pd.read_csv("processed/train_step1.csv")
test_df = pd.read_csv("processed/test_step1.csv")


In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# -----------------------------
# Load train and test CSVs
train_df = pd.read_csv(r"C:\Users\HP\smart-pricing\dataset\train.csv")  # adjust to your folder
test_df = pd.read_csv(r"C:\Users\HP\smart-pricing\dataset\test.csv")


# -----------------------------
# Function to safely get embedding
def process_url(url):
    try:
        return get_embedding(url).astype(np.float32)
    except:
        return np.zeros(2048, dtype=np.float32)  # fallback vector

# -----------------------------
# Function to process embeddings in batches with threads
def process_embeddings(urls, save_file, embedding_dim=2048, max_workers=16, batch_size=1000):
    # Ensure folder exists
    os.makedirs(os.path.dirname(save_file), exist_ok=True)
    save_file = os.path.abspath(save_file)  # Windows-safe absolute path

    n = len(urls)
    embeddings = np.zeros((n, embedding_dim), dtype=np.float32)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for start in tqdm(range(0, n, batch_size), total=(n + batch_size - 1) // batch_size, desc=os.path.basename(save_file)):
            end = min(start + batch_size, n)
            batch_urls = urls[start:end]
            # concurrent map
            batch_embeddings = list(executor.map(process_url, batch_urls))
            embeddings[start:end] = np.array(batch_embeddings, dtype=np.float32)

    np.save(save_file, embeddings)
    print(f"Saved embeddings to {save_file}")
    return embeddings

# -----------------------------
# Train embeddings
train_embeddings = process_embeddings(
    train_df['image_link'].tolist(),
    "processed/train_images_resnet.npy",
    embedding_dim=2048,
    max_workers=16,
    batch_size=1000
)

# Test embeddings
test_embeddings = process_embeddings(
    test_df['image_link'].tolist(),
    "processed/test_images_resnet.npy",
    embedding_dim=2048,
    max_workers=16,
    batch_size=1000
)


train_images_resnet.npy: 100%|██████████| 75/75 [00:02<00:00, 26.21it/s]


Saved embeddings to c:\Users\HP\smart-pricing\processed\train_images_resnet.npy


test_images_resnet.npy: 100%|██████████| 75/75 [00:03<00:00, 23.56it/s]


Saved embeddings to c:\Users\HP\smart-pricing\processed\test_images_resnet.npy


In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

# Load text TF-IDF features
tfidf_train = sparse.load_npz("processed/tfidf_train.npz")
tfidf_test = sparse.load_npz("processed/tfidf_test.npz")

# Load image embeddings
train_img = np.load("processed/train_images_resnet.npy")
test_img = np.load("processed/test_images_resnet.npy")

# Load original CSVs to access numeric/meta features
train = pd.read_csv("dataset/train.csv")
test = pd.read_csv("dataset/test.csv")


In [ ]:
import os
print(os.getcwd())


c:\Users\HP\smart-pricing


In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ["ipq", "size_val", "text_len", "num_tokens", "num_digits", "upper_ratio"]
scaler = StandardScaler()

train_num = scaler.fit_transform(train[num_cols].fillna(0))
test_num = scaler.transform(test[num_cols].fillna(0))


In [ ]:
print(train.columns.tolist())


['sample_id', 'catalog_content', 'image_link', 'price']


In [ ]:
print([f"'{c}'" for c in train.columns])


["'sample_id'", "'catalog_content'", "'image_link'", "'price'"]


In [ ]:
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()


In [ ]:
print(list(train.columns))


['sample_id', 'catalog_content', 'image_link', 'price']


In [ ]:
# Remove leading/trailing spaces and convert to lowercase
train.columns = train.columns.str.strip().str.lower()
test.columns = test.columns.str.strip().str.lower()


In [ ]:
num_cols = ["ipq", "size_val", "text_len", "num_tokens", "num_digits", "upper_ratio"]


In [ ]:
missing_cols = [c for c in num_cols if c not in train.columns]
print("Missing columns:", missing_cols)


Missing columns: ['ipq', 'size_val', 'text_len', 'num_tokens', 'num_digits', 'upper_ratio']


In [ ]:
print(train.shape)
print(train.head())


(75000, 4)
   sample_id                                    catalog_content  \
0      33127  Item Name: La Victoria Green Taco Sauce Mild, ...   
1     198967  Item Name: Salerno Cookies, The Original Butte...   
2     261251  Item Name: Bear Creek Hearty Soup Bowl, Creamy...   
3      55858  Item Name: Judee’s Blue Cheese Powder 11.25 oz...   
4     292686  Item Name: kedem Sherry Cooking Wine, 12.7 Oun...   

                                          image_link  price  
0  https://m.media-amazon.com/images/I/51mo8htwTH...   4.89  
1  https://m.media-amazon.com/images/I/71YtriIHAA...  13.12  
2  https://m.media-amazon.com/images/I/51+PFEe-w-...   1.97  
3  https://m.media-amazon.com/images/I/41mu0HAToD...  30.34  
4  https://m.media-amazon.com/images/I/41sA037+Qv...  66.49  


In [ ]:
print([repr(c) for c in train.columns])


["'sample_id'", "'catalog_content'", "'image_link'", "'price'"]


In [ ]:
train.columns = train.columns.str.strip().str.lower()
test.columns = test.columns.str.strip().str.lower()


In [ ]:
num_cols = ["ipq", "size_val", "text_len", "num_tokens", "num_digits", "upper_ratio"]


In [ ]:
num_cols = ["ipq", "size_val", "text_len", "num_tokens", "num_digits", "upper_ratio"]

# Create missing columns filled with zeros
for c in num_cols:
    if c not in train.columns:
        train[c] = 0
    if c not in test.columns:
        test[c] = 0


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_num = scaler.fit_transform(train[num_cols])
test_num = scaler.transform(test[num_cols])


In [ ]:
# Strip any extra whitespace in column names
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

num_cols = ["ipq", "size_val", "text_len", "num_tokens", "num_digits", "upper_ratio"]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

train_num = scaler.fit_transform(train[num_cols].fillna(0))
test_num = scaler.transform(test[num_cols].fillna(0))


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

# -----------------------------
# Load TF-IDF features
tfidf_train = sparse.load_npz("processed/tfidf_train.npz")
tfidf_test = sparse.load_npz("processed/tfidf_test.npz")

# Load image embeddings
train_img = np.load("processed/train_images_resnet.npy")
test_img = np.load("processed/test_images_resnet.npy")

# Load original CSVs
train = pd.read_csv("dataset/train.csv")
test = pd.read_csv("dataset/test.csv")

# -----------------------------
# Normalize column names
train.columns = train.columns.str.strip().str.lower()
test.columns = test.columns.str.strip().str.lower()

# -----------------------------
# Numeric/tabular features
num_cols = ["ipq", "size_val", "text_len", "num_tokens", "num_digits", "upper_ratio"]

# Create missing numeric columns as zeros
for c in num_cols:
    if c not in train.columns:
        train[c] = 0
    if c not in test.columns:
        test[c] = 0

# Fill NaNs with 0 and scale
scaler = StandardScaler()
train_num = scaler.fit_transform(train[num_cols].fillna(0))
test_num = scaler.transform(test[num_cols].fillna(0))

# Convert numeric features to sparse matrices
train_num_sparse = sparse.csr_matrix(train_num)
test_num_sparse = sparse.csr_matrix(test_num)

# -----------------------------
# Convert image embeddings to sparse (optional, keeps all features in sparse format)
train_img_sparse = sparse.csr_matrix(train_img)
test_img_sparse = sparse.csr_matrix(test_img)

# -----------------------------
# Combine all features: TF-IDF + Image + Numeric
X_train = hstack([tfidf_train, train_img_sparse, train_num_sparse])
X_test  = hstack([tfidf_test,  test_img_sparse,  test_num_sparse])

print("Train feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)


Train feature shape: (75000, 12054)
Test feature shape: (75000, 12054)


In [ ]:
from scipy import sparse

# Create folder if it doesn't exist
import os
os.makedirs("processed", exist_ok=True)

# Save sparse matrices
sparse.save_npz("processed/X_train_combined.npz", X_train)
sparse.save_npz("processed/X_test_combined.npz", X_test)

print("Saved combined features to processed/ folder")



Saved combined features to processed/ folder


In [ ]:
!pip install lightgbm



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from scipy import sparse
import pandas as pd

# Load combined features
X_train = sparse.load_npz("processed/X_train_combined.npz")
X_test  = sparse.load_npz("processed/X_test_combined.npz")

# Load train CSV to get target variable
train = pd.read_csv("dataset/train.csv")
y_train = train['price'].values  # target


In [ ]:
import numpy as np

def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error"""
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))


In [ ]:
from lightgbm import LGBMRegressor

# Create model
model = LGBMRegressor(
    objective='regression',
    learning_rate=0.05,
    n_estimators=500,
    num_leaves=31,
    n_jobs=-1,
    verbose=-1
)

# Fit model (supports verbose directly)
model.fit(X_train, y_train)


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [ ]:
import pandas as pd
import numpy as np
from scipy import sparse
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# Load combined features
X_train = sparse.load_npz("processed/X_train_combined.npz")
X_test  = sparse.load_npz("processed/X_test_combined.npz")

train = pd.read_csv("dataset/train.csv")
test  = pd.read_csv("dataset/test.csv")
y_train = train['price'].values

# Split train/validation manually
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Convert to LightGBM Dataset
lgb_train = lgb.Dataset(X_tr, label=y_tr)
lgb_val   = lgb.Dataset(X_val, label=y_val)

# Parameters
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1,
    'boosting_type': 'gbdt'
}

# Train model (without early stopping)
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=500  # fixed number of rounds
)

# -----------------------------
# Predict on validation and compute SMAPE
def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

y_val_pred = model.predict(X_val)
print("SMAPE on validation:", smape(y_val, y_val_pred))

# -----------------------------
# Predict on


SMAPE on validation: 61.131253163997435


In [ ]:
import pandas as pd
import numpy as np
from scipy import sparse
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import lightgbm as lgb

# -----------------------------
# Load features
X_tfidf_train = sparse.load_npz("processed/tfidf_train.npz")
X_tfidf_test  = sparse.load_npz("processed/tfidf_test.npz")

X_img_train = sparse.load_npz("processed/train_images_resnet.npz") if False else sparse.csr_matrix(np.load("processed/train_images_resnet.npy"))
X_img_test  = sparse.load_npz("processed/test_images_resnet.npz")  if False else sparse.csr_matrix(np.load("processed/test_images_resnet.npy"))

train = pd.read_csv("dataset/train.csv")
test  = pd.read_csv("dataset/test.csv")

# -----------------------------
# Numeric features
num_cols = ["ipq", "size_val", "text_len", "num_tokens", "num_digits", "upper_ratio"]
for c in num_cols:
    if c not in train.columns:
        train[c] = 0
    if c not in test.columns:
        test[c] = 0

scaler = StandardScaler()
train_num = scaler.fit_transform(train[num_cols].fillna(0))
test_num  = scaler.transform(test[num_cols].fillna(0))
train_num_sparse = sparse.csr_matrix(train_num)
test_num_sparse  = sparse.csr_matrix(test_num)

# -----------------------------
# Categorical features
cat_cols = ["brand1", "brand2", "size_unit"]
for c in cat_cols:
    if c not in train.columns:
        train[c] = ""
    if c not in test.columns:
        test[c] = ""

# Label encode
for c in cat_cols:
    le = LabelEncoder()
    combined = list(train[c].astype(str)) + list(test[c].astype(str))
    le.fit(combined)
    train[c] = le.transform(train[c].astype(str))
    test[c]  = le.transform(test[c].astype(str))

train_cat_sparse = sparse.csr_matrix(train[cat_cols].values)
test_cat_sparse  = sparse.csr_matrix(test[cat_cols].values)

# -----------------------------
# Combine all features: TF-IDF + Image + Numeric + Categorical
X_train_full = sparse.hstack([X_tfidf_train, X_img_train, train_num_sparse, train_cat_sparse])
X_test_full  = sparse.hstack([X_tfidf_test,  X_img_test,  test_num_sparse,  test_cat_sparse])

# -----------------------------
# Log-transform target
y_train = np.log1p(train['price'].values)

# -----------------------------
# Train/validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_train, test_size=0.2, random_state=42
)

# -----------------------------
# LightGBM parameters
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'verbose': -1,
    'boosting_type': 'gbdt'
}

# -----------------------------
# Convert to LightGBM Dataset
lgb_train = lgb.Dataset(X_tr, label=y_tr)
lgb_val   = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

# -----------------------------
# Train model
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=2000  # train more rounds
)

# -----------------------------
# Predict on validation and back-transform
y_val_pred_log = model.predict(X_val)
y_val_pred = np.expm1(y_val_pred_log)  # back to original price
y_val_true = np.expm1(y_val)          # original scale

# SMAPE function
def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

print("SMAPE on validation:", smape(y_val_true, y_val_pred))

# -----------------------------
# Predict on test and back-transform
y_test_pred_log = model.predict(X_test_full)
y_test_pred = np.expm1(y_test_pred_log)

# Save submission
submission = pd.DataFrame({
    'sample_id': test['sample_id'].values,
    'price': y_test_pred
})
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")


SMAPE on validation: 52.226741348358054
Submission saved to submission.csv


In [ ]:
# -----------------------------
# LightGBM Dataset (keep raw data for incremental training)
lgb_train = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
lgb_val   = lgb.Dataset(X_val, label=y_val, reference=lgb_train, free_raw_data=False)

# -----------------------------
# Parameters
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'verbose': -1,
    'boosting_type': 'gbdt'
}

# -----------------------------
# Manual early stopping
best_smape = float('inf')
best_iter  = 0
patience   = 5
wait       = 0
num_boost_round = 2000
chunk_size = 50

model = None
for start in range(0, num_boost_round, chunk_size):
    rounds = min(chunk_size, num_boost_round - start)
    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=rounds,
        init_model=model  # continue training from previous rounds
    )
    # Predict on validation
    y_val_pred_log = model.predict(X_val)
    y_val_pred = np.expm1(y_val_pred_log)
    y_val_true = np.expm1(y_val)
    smape_val = 100 * np.mean(2 * np.abs(y_val_pred - y_val_true) / (np.abs(y_val_pred) + np.abs(y_val_true) + 1e-8))
    print(f"Round {start + rounds}: SMAPE={smape_val:.4f}")

    # Early stopping logic
    if smape_val < best_smape:
        best_smape = smape_val
        best_iter = start + rounds
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"No improvement for {patience} rounds. Stopping early at iteration {best_iter}.")
            break


Round 50: SMAPE=59.5627
Round 100: SMAPE=56.6306
Round 150: SMAPE=55.5534
Round 200: SMAPE=54.9660
Round 250: SMAPE=54.5306
Round 300: SMAPE=54.2010
Round 350: SMAPE=53.9474
Round 400: SMAPE=53.6932
Round 450: SMAPE=53.5002
Round 500: SMAPE=53.3586
Round 550: SMAPE=53.2428
Round 600: SMAPE=53.1406
Round 650: SMAPE=53.0509
Round 700: SMAPE=52.9513
Round 750: SMAPE=52.8878
Round 800: SMAPE=52.8362
Round 850: SMAPE=52.7712
Round 900: SMAPE=52.7222
Round 950: SMAPE=52.6661
Round 1000: SMAPE=52.6122
Round 1050: SMAPE=52.5659
Round 1100: SMAPE=52.5385
Round 1150: SMAPE=52.5094
Round 1200: SMAPE=52.4755
Round 1250: SMAPE=52.4602
Round 1300: SMAPE=52.4280
Round 1350: SMAPE=52.3933
Round 1400: SMAPE=52.3922
Round 1450: SMAPE=52.3624
Round 1500: SMAPE=52.3506
Round 1550: SMAPE=52.3315
Round 1600: SMAPE=52.3296
Round 1650: SMAPE=52.3118
Round 1700: SMAPE=52.3053
Round 1750: SMAPE=52.2767
Round 1800: SMAPE=52.2843
Round 1850: SMAPE=52.2720
Round 1900: SMAPE=52.2540
Round 1950: SMAPE=52.2401
Round 

In [ ]:
import torch
print("Torch CUDA available:", torch.cuda.is_available())


Torch CUDA available: True


In [ ]:
params['device'] = 'gpu'
params['gpu_platform_id'] = 0
params['gpu_device_id'] = 0


In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.is_available())


NVIDIA GeForce RTX 3050 Laptop GPU
True


In [ ]:
import lightgbm as lgb
print("LightGBM built with GPU:", lgb.basic._ConfigAliases().get("device_type", None))


LightGBM built with GPU: {'device', 'device_type', None}


In [ ]:
params = {'device': 'gpu', 'objective': 'regression', 'metric': 'rmse'}


In [ ]:
# -----------------------------
# 0️⃣ Imports
# -----------------------------
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from sentence_transformers import SentenceTransformer
from torchvision import models, transforms
from PIL import Image
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# -----------------------------
# 1️⃣ Setup device
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("🔥 Using device:", device)

# -----------------------------
# 2️⃣ Load dataset
# -----------------------------
train_df = pd.read_csv("dataset/train.csv")
test_df = pd.read_csv("dataset/test.csv")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# -----------------------------
# 3️⃣ Text embeddings using SBERT
# -----------------------------
sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

def get_text_embeddings(texts):
    return sbert_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

train_text_emb = get_text_embeddings(train_df['catalog_content'].fillna('').tolist())
test_text_emb = get_text_embeddings(test_df['catalog_content'].fillna('').tolist())
print("Text embeddings shape:", train_text_emb.shape)

# -----------------------------
# 4️⃣ Image embeddings using ResNet50
# -----------------------------
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])  # Remove final FC
resnet.eval().to(device)

img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def load_image(url):
    try:
        img = Image.open(url).convert('RGB')
        img = img_transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = resnet(img).cpu().numpy().flatten()
        return emb
    except:
        return np.zeros(2048, dtype=np.float32)

train_img_emb = np.array([load_image(url) for url in tqdm(train_df['image_link'])])
test_img_emb = np.array([load_image(url) for url in tqdm(test_df['image_link'])])
print("Image embeddings shape:", train_img_emb.shape)

# -----------------------------
# 5️⃣ Tabular numeric features
# -----------------------------
train_df['text_len'] = train_df['catalog_content'].str.len()
train_df['num_digits'] = train_df['catalog_content'].str.count(r'\d')
test_df['text_len'] = test_df['catalog_content'].str.len()
test_df['num_digits'] = test_df['catalog_content'].str.count(r'\d')

num_cols = ['text_len', 'num_digits']
scaler = StandardScaler()
train_num = scaler.fit_transform(train_df[num_cols].fillna(0))
test_num = scaler.transform(test_df[num_cols].fillna(0))
print("Numeric features shape:", train_num.shape)

# -----------------------------
# 6️⃣ Combine all features
# -----------------------------
X_train = np.hstack([train_text_emb, train_img_emb, train_num])
X_test = np.hstack([test_text_emb, test_img_emb, test_num])
y = train_df['price'].values
print("Combined train shape:", X_train.shape)

# -----------------------------
# 7️⃣ Train/validation split
# -----------------------------
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y, test_size=0.1, random_state=42)

# -----------------------------
# 8️⃣ LightGBM GPU training
# -----------------------------
lgb_train = lgb.Dataset(X_tr, y_tr)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 255,
    'device': 'gpu',  # GPU
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'verbose': -1
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=50)
]

print("🚀 Training LightGBM on GPU...")
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=2000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train','val'],
    callbacks=callbacks
)

# -----------------------------
# 9️⃣ SMAPE evaluation
# -----------------------------
def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-9))

y_val_pred = model.predict(X_val)
print("SMAPE on validation:", smape(y_val, y_val_pred))

# -----------------------------
# 🔟 Predict test set and save
# -----------------------------
test_pred = model.predict(X_test)
submission = pd.DataFrame({'sample_id': test_df['sample_id'], 'price': test_pred})
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")


🔥 Using device: cuda
Train shape: (75000, 4)
Test shape: (75000, 3)


Batches: 100%|██████████| 2344/2344 [02:13<00:00, 17.59it/s]


Text embeddings shape: (75000, 384)


c:\Users\HP\smart-pricing\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\HP\smart-pricing\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 75000/75000 [00:02<00:00, 33218.11it/s]


Image embeddings shape: (75000, 2048)
Numeric features shape: (75000, 2)
Combined train shape: (75000, 2434)
🚀 Training LightGBM on GPU...
Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 23.9927	val's rmse: 28.3518
[100]	train's rmse: 19.9572	val's rmse: 27.6842
[150]	train's rmse: 17.3688	val's rmse: 27.4331
[200]	train's rmse: 15.5187	val's rmse: 27.2829
[250]	train's rmse: 13.9163	val's rmse: 27.1273
[300]	train's rmse: 12.6374	val's rmse: 27.0703
[350]	train's rmse: 11.6252	val's rmse: 27.0038
[400]	train's rmse: 10.8087	val's rmse: 26.993
[450]	train's rmse: 10.07	val's rmse: 26.9677
[500]	train's rmse: 9.41899	val's rmse: 26.9603
[550]	train's rmse: 8.85396	val's rmse: 26.9488
[600]	train's rmse: 8.33406	val's rmse: 26.9427
[650]	train's rmse: 7.88595	val's rmse: 26.94
[700]	train's rmse: 7.45232	val's rmse: 26.9362
Early stopping, best iteration is:
[693]	train's rmse: 7.49341	val's rmse: 26.9318
SMAPE on validation: 65.19484389389517
Submission s

In [ ]:
import os
print(os.path.exists("dataset/train.csv"))
print(os.path.exists("dataset/test.csv"))


True
True


In [ ]:
print(train_df.columns.tolist())


['sample_id', 'catalog_content', 'image_link', 'price']


In [ ]:
# -----------------------------
# SMAPE function (if not already defined)
# -----------------------------
def smape(y_true, y_pred):
    return 100 / len(y_true) * np.sum(
        2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-9)
    )

# -----------------------------
# Predict on validation
# -----------------------------
y_val_pred = model.predict(X_val)

# -----------------------------
# Compute SMAPE
# -----------------------------
smape_val = smape(y_val, y_val_pred)
print("SMAPE on validation:", smape_val)


SMAPE on validation: 65.19484389389517


In [ ]:
# -----------------------------
# 0️⃣ Imports
# -----------------------------
import numpy as np
import pandas as pd
from scipy import sparse
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack

# -----------------------------
# 1️⃣ Load Data and Features
# -----------------------------
train = pd.read_csv("dataset/train.csv")
test = pd.read_csv("dataset/test.csv")

# -----------------------------
# 2️⃣ Create Numeric Features
# -----------------------------
# Text-based features
train['text_len'] = train['catalog_content'].str.len()
test['text_len'] = test['catalog_content'].str.len()

train['num_digits'] = train['catalog_content'].str.count(r'\d')
test['num_digits'] = test['catalog_content'].str.count(r'\d')

train['upper_ratio'] = train['catalog_content'].apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x)+1e-8))
test['upper_ratio'] = test['catalog_content'].apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x)+1e-8))

# If you have numeric sizes in your text, extract here. Otherwise, fill with 0
train['size_val'] = 0
test['size_val'] = 0

num_cols = ["size_val", "text_len", "num_digits", "upper_ratio"]

scaler = StandardScaler()
train_num = scaler.fit_transform(train[num_cols].fillna(0))
test_num = scaler.transform(test[num_cols].fillna(0))

# -----------------------------
# 3️⃣ Load TF-IDF and Image Features
# -----------------------------
tfidf_train = sparse.load_npz("processed/tfidf_train.npz")
tfidf_test = sparse.load_npz("processed/tfidf_test.npz")

train_img = np.load("processed/train_images_resnet.npy")
test_img = np.load("processed/test_images_resnet.npy")

# Normalize image embeddings
train_img /= np.linalg.norm(train_img, axis=1, keepdims=True) + 1e-8
test_img /= np.linalg.norm(test_img, axis=1, keepdims=True) + 1e-8

# -----------------------------
# 4️⃣ Combine All Features
# -----------------------------
X_full = hstack([tfidf_train, sparse.csr_matrix(train_num), sparse.csr_matrix(train_img)])
y_full = train['price'].values

X_test_full = hstack([tfidf_test, sparse.csr_matrix(test_num), sparse.csr_matrix(test_img)])

# -----------------------------
# 5️⃣ Train/Validation Split
# -----------------------------
X_tr, X_val, y_tr, y_val = train_test_split(X_full, y_full, test_size=0.1, random_state=42)

# Log-transform target to handle skewed prices
y_tr_log = np.log1p(y_tr)
y_val_log = np.log1p(y_val)

# -----------------------------
# 6️⃣ SMAPE Function
# -----------------------------
def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

# -----------------------------
# 7️⃣ LightGBM Dataset
# -----------------------------
lgb_train = lgb.Dataset(X_tr, y_tr_log, free_raw_data=False)
lgb_val = lgb.Dataset(X_val, y_val_log, reference=lgb_train, free_raw_data=False)

# -----------------------------
# 8️⃣ LightGBM GPU Parameters
# -----------------------------
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,
    'num_leaves': 31,
    'max_depth': -1,
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'verbose': -1
}

# -----------------------------
# 9️⃣ Train LightGBM in Chunks
# -----------------------------
print("🚀 Training LightGBM on GPU...")
num_boost_round = 2000
chunk_size = 50

model = None
for start in range(0, num_boost_round, chunk_size):
    rounds = min(chunk_size, num_boost_round - start)
    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=rounds,
        valid_sets=[lgb_train, lgb_val],
        valid_names=['train', 'val'],
        init_model=model
    )
    y_val_pred_log = model.predict(X_val)
    y_val_pred = np.expm1(y_val_pred_log)
    score = smape(y_val, y_val_pred)
    print(f"Round {start + rounds}: SMAPE={score:.4f}")

# -----------------------------
# 🔟 Final Validation SMAPE
# -----------------------------
y_val_pred_log = model.predict(X_val)
y_val_pred = np.expm1(y_val_pred_log)
final_smape = smape(y_val, y_val_pred)
print(f"✅ Final SMAPE on validation: {final_smape:.4f}")

# -----------------------------
# 1️⃣1️⃣ Predict on Test Set & Save Submission
# -----------------------------
y_test_pred_log = model.predict(X_test_full)
y_test_pred = np.expm1(y_test_pred_log)

submission = pd.DataFrame({
    'sample_id': test['sample_id'],
    'price': y_test_pred
})
submission.to_csv("submission.csv", index=False)
print("✅ Submission saved to submission.csv")


🚀 Training LightGBM on GPU...
Round 50: SMAPE=68.2807
Round 100: SMAPE=65.1270
Round 150: SMAPE=63.3050
Round 200: SMAPE=61.9895
Round 250: SMAPE=61.0164
Round 300: SMAPE=60.2454
Round 350: SMAPE=59.6428
Round 400: SMAPE=59.1593
Round 450: SMAPE=58.7463
Round 500: SMAPE=58.3717
Round 550: SMAPE=58.0452
Round 600: SMAPE=57.7586
Round 650: SMAPE=57.5099
Round 700: SMAPE=57.3104
Round 750: SMAPE=57.1109
Round 800: SMAPE=56.9470
Round 850: SMAPE=56.8002
Round 900: SMAPE=56.6609
Round 950: SMAPE=56.5321
Round 1000: SMAPE=56.4106
Round 1050: SMAPE=56.2969
Round 1100: SMAPE=56.1905
Round 1150: SMAPE=56.0954
Round 1200: SMAPE=56.0002
Round 1250: SMAPE=55.9228
Round 1300: SMAPE=55.8363
Round 1350: SMAPE=55.7520
Round 1400: SMAPE=55.6727
Round 1450: SMAPE=55.5950
Round 1500: SMAPE=55.5192
Round 1550: SMAPE=55.4517
Round 1600: SMAPE=55.3809
Round 1650: SMAPE=55.3189
Round 1700: SMAPE=55.2646
Round 1750: SMAPE=55.2066
Round 1800: SMAPE=55.1481
Round 1850: SMAPE=55.0915
Round 1900: SMAPE=55.0314
Ro

In [1]:
# -----------------------------
# 0️⃣ Imports
# -----------------------------
import numpy as np
import pandas as pd
from scipy import sparse
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from torchvision import models, transforms
import torch
from PIL import Image
import requests
from io import BytesIO
import re
import os

# -----------------------------
# 1️⃣ Load Data
# -----------------------------
train = pd.read_csv("dataset/train.csv")
test = pd.read_csv("dataset/test.csv")

# -----------------------------
# 2️⃣ Extract Price per Unit
# -----------------------------
def extract_value_unit(text):
    try:
        val_match = re.search(r'Value:\s*([\d.]+)', str(text))
        unit_match = re.search(r'Unit:\s*(\w+)', str(text))
        val = float(val_match.group(1)) if val_match else np.nan
        unit = unit_match.group(1).lower() if unit_match else None
        if unit == 'ml': val /= 29.5735
        elif unit in ['g', 'gram', 'grams']: val /= 28.3495
        return val
    except:
        return np.nan

train['unit_value'] = train['catalog_content'].apply(extract_value_unit)
test['unit_value'] = test['catalog_content'].apply(extract_value_unit)

train['price_per_unit'] = train['price'] / train['unit_value']
train['price_per_unit'].replace([np.inf, -np.inf], np.nan, inplace=True)
test['price_per_unit'] = np.nan

# -----------------------------
# 3️⃣ Numeric + Text Features
# -----------------------------
scaler = StandardScaler()
num_cols = ["price_per_unit"]
train_num = scaler.fit_transform(train[num_cols].fillna(0))
test_num = scaler.transform(test[num_cols].fillna(0))

tfidf = TfidfVectorizer(max_features=3000)  # smaller = faster
tfidf_train = tfidf.fit_transform(train['catalog_content'].fillna(''))
tfidf_test = tfidf.transform(test['catalog_content'].fillna(''))

# -----------------------------
# 4️⃣ Fast Image Embeddings
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.Compose([
    transforms.Resize((128, 128)),  # smaller size = faster
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# ✅ Option: only one model (EfficientNet) for speed
effnet_model = models.efficientnet_b0(pretrained=True)
effnet_model = torch.nn.Sequential(*(list(effnet_model.children())[:-1])).to(device)
effnet_model.eval()

def get_image_features(urls, cache_name):
    cache_path = f"{cache_name}.npy"
    if os.path.exists(cache_path):
        print(f"⚡ Loading cached features from {cache_path}")
        return np.load(cache_path)

    n = len(urls)
    features = np.zeros((n, 1280), dtype=np.float32)
    for i, u in enumerate(urls):
        try:
            if str(u).startswith('http'):
                r = requests.get(u, timeout=3)
                img = Image.open(BytesIO(r.content)).convert("RGB")
            else:
                img = Image.open(u).convert("RGB")

            x = transform(img).unsqueeze(0).to(device)
            with torch.no_grad():
                eff_feat = effnet_model(x).cpu().numpy().flatten()
                eff_feat /= np.linalg.norm(eff_feat) + 1e-8
            features[i] = eff_feat
        except:
            features[i] = 0

        if (i+1) % 200 == 0:
            print(f"Processed {i+1}/{n} images")

    np.save(cache_path, features)
    print(f"✅ Saved features to {cache_path}")
    return features

print("Extracting EfficientNet features...")
#train_img = get_image_features(train['image_link'], "train_img_effnet")
#test_img = get_image_features(test['image_link'], "test_img_effnet")
# -----------------------------
# 4️⃣ Load Cached Image Features (Skip Reprocessing)
# -----------------------------
def safe_load_features(cache_name, expected_len, dim=1280):
    cache_path = f"{cache_name}.npy"
    if os.path.exists(cache_path):
        print(f"⚡ Loading existing image features: {cache_path}")
        arr = np.load(cache_path)
        if arr.shape[0] < expected_len:
            print(f"⚠️ Only {arr.shape[0]} / {expected_len} processed — filling rest with zeros.")
            pad = np.zeros((expected_len - arr.shape[0], dim), dtype=np.float32)
            arr = np.vstack([arr, pad])
        elif arr.shape[0] > expected_len:
            arr = arr[:expected_len]
    else:
        print(f"❌ No cache found for {cache_name}, using zeros.")
        arr = np.zeros((expected_len, dim), dtype=np.float32)
    return arr

# Use partial cached data if available
train_img = safe_load_features("train_img_effnet", len(train))
test_img = safe_load_features("test_img_effnet", len(test))


# -----------------------------
# 5️⃣ Combine all features
# -----------------------------
X_full = sparse.hstack([
    tfidf_train,
    sparse.csr_matrix(train_num),
    sparse.csr_matrix(train_img)
])
y_full = train['price'].values

X_test_full = sparse.hstack([
    tfidf_test,
    sparse.csr_matrix(test_num),
    sparse.csr_matrix(test_img)
])

# -----------------------------
# 6️⃣ Train LightGBM
# -----------------------------
X_tr, X_val, y_tr, y_val = train_test_split(X_full, y_full, test_size=0.1, random_state=42)
y_tr_log = np.log1p(y_tr)
y_val_log = np.log1p(y_val)

lgb_train = lgb.Dataset(X_tr, y_tr_log)
lgb_val = lgb.Dataset(X_val, y_val_log, reference=lgb_train)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'verbosity': -1,
    'device': 'gpu'
}

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

# -----------------------------
# 7️⃣ Predict Test
# -----------------------------
y_test_pred_log = model.predict(X_test_full)
y_test_pred = np.expm1(y_test_pred_log)
pd.DataFrame({'sample_id': test['sample_id'], 'price': y_test_pred}).to_csv("submission_fast.csv", index=False)
print("✅ submission_fast.csv saved")


C:\Users\HP\AppData\Local\Temp\ipykernel_17116\850887293.py:44: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['price_per_unit'].replace([np.inf, -np.inf], np.nan, inplace=True)
c:\Users\HP\smart-pricing\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\HP\smart-pricing\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a w

Extracting EfficientNet features...
❌ No cache found for train_img_effnet, using zeros.
❌ No cache found for test_img_effnet, using zeros.
Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 0.647823	val's rmse: 0.66058
[100]	train's rmse: 0.589575	val's rmse: 0.61152
[150]	train's rmse: 0.562768	val's rmse: 0.591423
[200]	train's rmse: 0.545728	val's rmse: 0.580306
[250]	train's rmse: 0.533274	val's rmse: 0.573918
[300]	train's rmse: 0.523048	val's rmse: 0.568786
[350]	train's rmse: 0.514275	val's rmse: 0.564838
[400]	train's rmse: 0.506146	val's rmse: 0.56183
[450]	train's rmse: 0.498964	val's rmse: 0.559459
[500]	train's rmse: 0.491907	val's rmse: 0.556781
Did not meet early stopping. Best iteration is:
[499]	train's rmse: 0.492022	val's rmse: 0.556755
✅ submission_fast.csv saved


In [3]:
import numpy as np

def smape(y_true, y_pred):
    """
    Compute Symmetric Mean Absolute Percentage Error (SMAPE).
    Formula: SMAPE = (100 / n) * Σ ( |y_pred - y_true| / ((|y_true| + |y_pred|)/2) )
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = np.abs(y_pred - y_true) / np.maximum(denominator, 1e-8)
    return 100 * np.mean(diff)


In [4]:
# Predictions for validation set
y_val_pred_log = model.predict(X_val)
y_val_pred = np.expm1(y_val_pred_log)

# Compute SMAPE on validation set
val_smape = smape(y_val, y_val_pred)
print(f"Validation SMAPE: {val_smape:.4f}")


Validation SMAPE: 41.9831


In [5]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 64,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'device': 'gpu'
}


In [6]:
y_test_pred = np.clip(y_test_pred, 0, np.percentile(y_test_pred, 99))


In [7]:
model = lgb.train(params, lgb.Dataset(X_full, np.log1p(y_full)), num_boost_round=model.best_iteration)


In [8]:
import numpy as np

def smape(y_true, y_pred):
    """
    Compute Symmetric Mean Absolute Percentage Error (SMAPE).
    Formula: SMAPE = (100 / n) * Σ ( |y_pred - y_true| / ((|y_true| + |y_pred|)/2) )
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = np.abs(y_pred - y_true) / np.maximum(denominator, 1e-8)
    return 100 * np.mean(diff)


In [9]:
# Predictions for validation set
y_val_pred_log = model.predict(X_val)
y_val_pred = np.expm1(y_val_pred_log)

# Compute SMAPE on validation set
val_smape = smape(y_val, y_val_pred)
print(f"Validation SMAPE: {val_smape:.4f}")


Validation SMAPE: 35.9840


In [10]:
# Predict on test set
y_test_pred_log = model.predict(X_test_full)
y_test_pred = np.expm1(y_test_pred_log)

# Optional: clip extreme predictions
y_test_pred = np.clip(y_test_pred, 0, np.percentile(y_test_pred, 99))

# Save submission CSV
submission_file = "submission_final.csv"
submission_final = pd.DataFrame({
    'sample_id': test['sample_id'],
    'price': y_test_pred
})
submission_final.to_csv(submission_file, index=False)
print(f"✅ Submission saved: {submission_file}")


✅ Submission saved: submission_final.csv


In [11]:
import zipfile
import os

# Folder containing your code
folder_to_zip = "smart-pricing"
zip_filename = "smart_pricing_code.zip"

# Allowed file types
allowed_extensions = ['.ipynb', '.py', '.txt', '.md']  # include notebooks and scripts only

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder_to_zip):
        for file in files:
            if os.path.splitext(file)[1] in allowed_extensions:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_to_zip)
                zipf.write(file_path, arcname)
